In [1]:
import pandas as pd
import numpy as np

# List all the CSV files we want to investigate
files = ['payouts.csv', 'country_master.csv', 'processors.csv', 'fx_rates.csv']

# Loop through each file one by one to discover what is inside
for file in files:
    print("\n" + "="*50)
    print(f"🔍 INVESTIGATING FILE: {file}")
    print("="*50)
    
    # 1. Load the file into a DataFrame
    df = pd.read_csv(file)
    
    # 2. How big is this data? Check rows and columns
    print(f"🔹 Data Size: {df.shape[0]} rows and {df.shape[1]} columns")
    
    # 3. What are the column names and how is Python reading them?
    print("\n🔹 Column Names & Their Data Types:")
    print(df.dtypes)
    
    # 4. Let me see a quick preview of the actual data rows
    print("\n🔹 First 3 Rows Preview:")
    print(df.head(3))
    
    # 5. Are there any blank/missing values we need to worry about?
    print("\n🔹 Missing Values Check:")
    missing_counts = df.isnull().sum()
    if missing_counts.sum() > 0:
        # Show only the columns that actually have missing data
        print(missing_counts[missing_counts > 0])
    else:
        print("Perfect! No missing values in this file.")
        
    # 6. For text columns, what kind of categories exist inside them?
    print("\n🔹 Text Column Categories:")
    # Select only the text/object columns
    text_cols = df.select_dtypes(include=['object']).columns
    
    for col in text_cols:
        unique_count = df[col].nunique()
        print(f"   - Column '{col}' has {unique_count} unique categories")
        
        # If the number of categories is small, let's look at the actual names inside
        if unique_count <= 20:
            print(f"     -> Values look like: {df[col].unique()}")
            
            


🔍 INVESTIGATING FILE: payouts.csv
🔹 Data Size: 40000 rows and 20 columns

🔹 Column Names & Their Data Types:
payout_id               object
payout_datetime         object
sender_country          object
receiver_country        object
sender_currency         object
receiver_currency       object
amount_sent            float64
customer_type           object
payment_channel         object
payment_type            object
processor_id            object
expected_fx_rate       float64
actual_fx_rate         float64
sla_target_minutes       int64
compliance_review       object
payout_fee             float64
processing_minutes       int64
settlement_datetime     object
payout_status           object
failure_reason          object
dtype: object

🔹 First 3 Rows Preview:
   payout_id payout_datetime sender_country receiver_country sender_currency  \
0  PAY100000      2026-11-24        Germany            India             EUR   
1  PAY100001      2026-10-07             UK           Canada           

Perfect! No missing values in this file.

🔹 Text Column Categories:
   - Column 'date' has 365 unique categories
   - Column 'currency' has 19 unique categories
     -> Values look like: ['USD' 'EUR' 'GBP' 'INR' 'AED' 'SAR' 'QAR' 'SGD' 'JPY' 'AUD' 'MYR' 'CNY'
 'BRL' 'MXN' 'ZAR' 'KES' 'NGN' 'CAD' 'IDR']
   - Column 'volatility' has 3 unique categories
     -> Values look like: ['Low' 'Medium' 'High']


In [2]:
payouts = pd.read_csv('payouts.csv')

# Let's sort all unique sender countries alphabetically so variations sit next to each other
print("--- Alphabetical List of Sender Countries ---")
unique_senders = sorted(payouts['sender_country'].unique())

# Print them out to inspect with our own eyes
for country in unique_senders:
    print(country)

--- Alphabetical List of Sender Countries ---
AUSTRALIA
Australia
BRAZIL
Brazil
CANADA
CHINA
Canada
China
FRANCE
France
GERMANY
Germany
INDIA
INDONESIA
India
Indonesia
JAPAN
Japan
KENYA
Kenya
MALAYSIA
MEXICO
Malaysia
Mexico
NIGERIA
Nigeria
QATAR
Qatar
SAUDI ARABIA
SINGAPORE
SOUTH AFRICA
Saudi Arabia
Singapore
South Africa
UAE
UK
USA
australia
brazil
canada
china
france
germany
india
indonesia
japan
kenya
malaysia
mexico
nigeria
qatar
saudi arabia
singapore
south africa
uae
uk
usa


## Data Cleaning

In [3]:
# 1. Load raw data
payouts = pd.read_csv('payouts.csv')
fx_rates = pd.read_csv('fx_rates.csv')
country_masters = pd.read_csv('country_master.csv')
processors = pd.read_csv('processors.csv')


# --- CLEANING PAYOUTS.CSV ---

# Fix 1: Fix text casing in sender_country (reduces 57 messy variations down to 20 clean countries)
payouts['sender_country'] = payouts['sender_country'].str.title()
payouts['receiver_country'] = payouts['receiver_country'].str.title()

acronym_fixes = {'Uk': 'UK', 'Uae': 'UAE', 'Usa': 'USA'}
payouts['sender_country'] = payouts['sender_country'].replace(acronym_fixes)
payouts['receiver_country'] = payouts['receiver_country'].replace(acronym_fixes)




# Fix 2: Fill missing payout_fee values 
# 1. Calculate the fee rate for each row where data exists
payouts['fee_rate'] = payouts['payout_fee'] / payouts['amount_sent']

# 2. Get the exact fee rate per processor_id
processor_rates = payouts.groupby('processor_id')['fee_rate'].mean()

# 3. Fill missing payout_fee values accurately
payouts['payout_fee'] = payouts['payout_fee'].fillna(
    payouts['amount_sent'] * payouts['processor_id'].map(processor_rates))

# 4. Get overall average fee rate across the whole company (as a fallback)
overall_avg_rate = payouts['fee_rate'].mean()

# 5. Fallback fill for rows where processor_id was ALSO missing
payouts['payout_fee'] = payouts['payout_fee'].fillna(
    payouts['amount_sent'] * overall_avg_rate)

# Clean up temporary column
payouts.drop(columns=['fee_rate'], inplace=True)



# Fix 3: Fill missing processor_id values
payouts['processor_id'] = payouts['processor_id'].fillna('UNKNOWN')



# Fix 4: Handle missing failure_reason logically
payouts.loc[payouts['payout_status'] == 'Success', 'failure_reason'] = payouts['failure_reason'].fillna('Not Applicable')
payouts.loc[payouts['payout_status'] == 'Failed', 'failure_reason'] = payouts['failure_reason'].fillna('Unknown Error')



# Fix 5: Convert string dates to actual Datetime format
payouts['payout_datetime'] = pd.to_datetime(payouts['payout_datetime'])
payouts['settlement_datetime'] = pd.to_datetime(payouts['settlement_datetime'])


# For failed payouts, keep settlement_datetime explicitly empty (NaT)
failed_mask = payouts['payout_status'] == 'Failed'
payouts.loc[failed_mask, 'settlement_datetime'] = pd.NaT

# --- CLEANING FX_RATES.CSV ---

# Fix 6: Convert FX date column to Datetime format
fx_rates['date'] = pd.to_datetime(fx_rates['date'])




print("=== 1. FIRST 10 ROWS OF CLEANED PAYOUTS ===")
print(payouts.head(10))

print("\n=== 2. MISSING VALUE COUNT (Should all be 0) ===")
print(payouts.isna().sum())

print("\n=== 3. DATATYPES AND NON-NULL COUNTS ===")
print(payouts.info())

print("\n=== 4. NUMERICAL SUMMARY (Check payout_fee stats) ===")
print(payouts[['amount_sent', 'payout_fee']].describe())





# --- SAVE CLEANED FILES ---
#payouts.to_csv('cleaned_payouts.csv', index=False)
fx_rates.to_csv('cleaned_fx_rates.csv', index=False)


print("Data cleaning completed! Cleaned files saved successfully.")

=== 1. FIRST 10 ROWS OF CLEANED PAYOUTS ===
   payout_id payout_datetime sender_country receiver_country sender_currency  \
0  PAY100000      2026-11-24        Germany            India             EUR   
1  PAY100001      2026-10-07             UK           Canada             GBP   
2  PAY100002      2026-01-14        Nigeria     Saudi Arabia             NGN   
3  PAY100003      2026-05-23          India              UAE             INR   
4  PAY100004      2026-06-22        Germany               UK             EUR   
5  PAY100005      2026-01-23         Mexico          Nigeria             MXN   
6  PAY100006      2026-11-18      Indonesia         Malaysia             IDR   
7  PAY100007      2026-05-29             UK            Qatar             GBP   
8  PAY100008      2026-03-25       Malaysia     Saudi Arabia             MYR   
9  PAY100009      2026-11-22            UAE          Nigeria             AED   

  receiver_currency  amount_sent customer_type payment_channel  \
0        

## Feature Engineering


In [4]:
# 1. Payment Corridor
payouts['payment_corridor'] = payouts['sender_country'] + ' -> ' + payouts['receiver_country']

# 2. FX Slippage Percentage & USD Impact
payouts['fx_slippage'] = ((payouts['actual_fx_rate'] - payouts['expected_fx_rate']) / payouts['expected_fx_rate'])
payouts['fx_slippage_pct'] = payouts['fx_slippage'] * 100
payouts['fx_profit_loss_usd'] = payouts['fx_slippage'] * payouts['payout_fee']

# 3. SLA Compliance Flag 
payouts['sla_met'] = np.where(payouts['processing_minutes'] <= payouts['sla_target_minutes'], 'Yes', 'No')

# 4. Effective Fee Percentage
payouts['effective_fee_pct'] = (payouts['payout_fee'] / payouts['amount_sent']) * 100

# 5. Extract Time Features
payouts['payout_month'] = payouts['payout_datetime'].dt.month_name()
payouts['payout_day_of_week'] = payouts['payout_datetime'].dt.day_name()

new_cols = [
    'payment_corridor', 'fx_slippage_pct', 'fx_profit_loss_usd',
    'sla_met', 'effective_fee_pct', 'payout_month', 'payout_day_of_week'
]

print(payouts[new_cols].head(10))

# Save final dataset ready for Dashboarding / SQL / EDA
payouts.to_csv('final_payouts.csv', index=False)

print("Feature engineering complete! Saved as 'final_payouts.csv'.")


           payment_corridor  fx_slippage_pct  fx_profit_loss_usd sla_met  \
0          Germany -> India        -0.075317           -1.390326     Yes   
1              UK -> Canada         0.046427           27.439234     Yes   
2   Nigeria -> Saudi Arabia        -0.068796      -201340.245571      No   
3              India -> UAE        -0.088382        -4036.095286     Yes   
4             Germany -> UK         0.020237           13.409832     Yes   
5         Mexico -> Nigeria        -0.579957           -4.227544      No   
6     Indonesia -> Malaysia         0.199466       324280.382568     Yes   
7               UK -> Qatar        -0.063405         -171.286576      No   
8  Malaysia -> Saudi Arabia        -0.039174           -8.761576     Yes   
9            UAE -> Nigeria        -0.075443           -2.969618      No   

   effective_fee_pct payout_month payout_day_of_week  
0           0.449999     November            Tuesday  
1           0.520000      October          Wednesday 

## SQL Analysis

In [5]:
import sqlite3

# 1. Create an in-memory SQLite database connection
conn = sqlite3.connect(":memory:")

# 2. Load CSV files into SQLite database tables
country_masters.to_sql(
    "country_master", conn, index=False, if_exists="replace"
)
payouts.to_sql(
    "payouts", conn, index=False, if_exists="replace"
)
fx_rates.to_sql(
    "fx_rates", conn, index=False, if_exists="replace"
)
processors.to_sql(
    "processors", conn, index=False, if_exists="replace"
)

print("Tables created successfully! Ready for SQL queries.")

Tables created successfully! Ready for SQL queries.


### Business Question 1: Which Customer Segment Generates the Highest Transaction Volume and Fee Revenue?

### Objective

This analysis compares different customer segments (Retail, SME, and Corporate) based on:

- Total number of payout transactions
- Total payout amount sent
- Total fee revenue generated

The objective is to identify the most valuable customer segment and understand which segment contributes the most to transaction volume and revenue.

In [6]:
def format_number(num):
    if num >= 1_000_000_000:
        return f"{num/1_000_000_000:.2f} B"
    elif num >= 1_000_000:
        return f"{num/1_000_000:.2f} M"
    elif num >= 1_000:
        return f"{num/1_000:.2f} K"
    else:
        return str(num)


query_1 = """
SELECT 
    customer_type,
    COUNT(payout_id) AS total_transactions,
    ROUND(SUM(amount_sent), 2) AS total_volume_sent,
    ROUND(SUM(payout_fee), 2) AS total_fee_revenue
FROM payouts
GROUP BY customer_type;
"""
df_q1 = pd.read_sql(query_1, conn)

df_q1["total_volume_sent"] = df_q1["total_volume_sent"].apply(format_number)
df_q1["total_fee_revenue"] = df_q1["total_fee_revenue"].apply(format_number)

df_q1

,customer_type,total_transactions,total_volume_sent,total_fee_revenue
0,Corporate,4025,21.03 B,113.92 M
1,Retail,28002,2.85 B,15.49 M
2,SME,7973,6.21 B,33.75 M


### Key Insights

- Retail customers performed the highest number of transactions.
- Corporate customers generated the highest payout volume.
- Corporate customers also contributed the highest fee revenue despite having fewer transactions than Retail customers.
- This indicates that Corporate customers perform higher-value transactions on average.

### Business Question 2: Which Payment Processor Has the Highest SLA Breach Rate?

### Objective

Analyze the performance of each payment processor by comparing average processing time with the promised SLA and identifying SLA breaches.

In [7]:
query_2 = """
SELECT 
    pr.processor_name,
    pr.sla_minutes AS promised_sla_min,
    ROUND(AVG(p.processing_minutes), 2) AS avg_actual_processing_min,
    SUM(CASE WHEN p.sla_met = 'No' THEN 1 ELSE 0 END) AS sla_breaches,
    ROUND(100.0 * SUM(CASE WHEN p.sla_met = 'No' THEN 1 ELSE 0 END) / COUNT(*), 2) AS breach_rate_pct
FROM payouts p
JOIN processors pr 
    ON p.processor_id = pr.processor_id
GROUP BY pr.processor_name, pr.sla_minutes
ORDER BY breach_rate_pct DESC;
"""
df_q2 = pd.read_sql(query_2, conn)

df_q2

,processor_name,promised_sla_min,avg_actual_processing_min,sla_breaches,breach_rate_pct
0,AfricaConnect,40,66.84,1578,80.47
1,AsiaFast,18,37.28,3434,74.88
2,TrustTransfer,28,50.67,2788,70.32
3,AmeriTrans,24,42.68,1764,69.39
4,GlobalWire,25,46.24,4528,68.75
5,SwiftPay,20,38.25,7738,68.65
6,FinBridge,23,39.54,3022,65.16
7,MiddleEastFlow,20,35.02,1313,64.84
8,EuroLink,22,32.81,1041,51.38


### Key Insights

- AfricaConnect has the highest SLA breach rate (80.47%).
- EuroLink has the lowest SLA breach rate (51.38%).
- Most processors exceed their promised SLA, indicating operational delays.
- High breach rates may negatively impact customer experience and service reliability.

### Business Question 3: Which Payment Corridors Have the Highest Payout Failure Rate?

### Objective

Analyze payout failure rates across different payment corridors to identify routes with frequent transaction failures and potential operational issues.

In [8]:
query_3 = """
SELECT 
    payment_corridor,
    COUNT(*) AS total_payouts,
    SUM(CASE WHEN payout_status = 'Failed' THEN 1 ELSE 0 END) AS failed_payouts,
    ROUND(100.0 * SUM(CASE WHEN payout_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS failure_rate_pct
FROM payouts
GROUP BY payment_corridor
HAVING total_payouts > 50
ORDER BY failure_rate_pct DESC;
"""
df_q3 = pd.read_sql(query_3, conn)

df_q3

,payment_corridor,total_payouts,failed_payouts,failure_rate_pct
0,India -> Nigeria,103,18,17.48
1,Canada -> Nigeria,93,16,17.20
2,Saudi Arabia -> Nigeria,109,17,15.60
3,Japan -> Nigeria,101,15,14.85
4,Australia -> Nigeria,116,17,14.66
...,...,...,...,...
375,Brazil -> Singapore,108,0,0.00
376,Brazil -> Malaysia,115,0,0.00
377,Brazil -> Japan,94,0,0.00
378,Australia -> UAE,93,0,0.00


### Key Insights

- India → Nigeria has the highest payout failure rate (17.48%).
- Several corridors have a 0% failure rate, indicating stable transaction processing.
- High-failure corridors should be investigated to identify the root cause and improve success rates.

### Business Question 4: Which Payment Processor Handles the Highest Transaction Volume in Each Region?

### Objective

Compare payment processors across different regions based on transaction volume, total transactions, and success rate to identify the top-performing processor in each region.

In [9]:
query_4 = """
WITH processor_metrics AS (
    SELECT 
        pr.specialization_region,
        pr.processor_name,
        COUNT(p.payout_id) AS total_txns,
        ROUND(SUM(p.amount_sent), 2) AS total_volume,
        ROUND(100.0 * SUM(CASE WHEN p.payout_status = 'Success' THEN 1 ELSE 0 END) / COUNT(p.payout_id), 2) AS success_rate
    FROM processors pr
    JOIN payouts p ON pr.processor_id = p.processor_id
    GROUP BY pr.specialization_region, pr.processor_name
)
SELECT 
    specialization_region,
    processor_name,
    total_txns,
    total_volume,
    success_rate,
    DENSE_RANK() OVER (PARTITION BY specialization_region ORDER BY total_volume DESC) AS region_volume_rank
FROM processor_metrics;
"""
df_q4 = pd.read_sql(query_4, conn)

df_q4["total_volume"] = df_q4["total_volume"].apply(format_number)

df_q4

,specialization_region,processor_name,total_txns,total_volume,success_rate,region_volume_rank
0,APAC,AsiaFast,4586,3.49 B,97.58,1
1,Africa,AfricaConnect,1961,1.41 B,89.44,1
2,Americas,AmeriTrans,2542,1.97 B,96.62,1
3,Europe,EuroLink,2026,1.57 B,98.08,1
4,Global,SwiftPay,11271,8.32 B,97.10,1
5,Global,GlobalWire,6586,4.83 B,96.22,2
6,Global,FinBridge,4638,3.59 B,97.87,3
7,Global,TrustTransfer,3965,3.06 B,95.91,4
8,Middle East,MiddleEastFlow,2025,1.58 B,98.12,1


### Key Insights

- SwiftPay ranks first globally with the highest transaction volume (8.32 B).
- AsiaFast, EuroLink, AmeriTrans, AfricaConnect, and MiddleEastFlow lead their respective regions.
- Most processors maintain a success rate above 95%, indicating reliable payment processing.

### Business Question 5: How Do Payout Failure Rates Vary by Region and AML Risk?

### Objective

Analyze payout performance across different regions and AML risk levels to understand whether higher-risk regions experience more transaction failures.

In [10]:
query_5 = """
SELECT 
    c.region,
    c.aml_risk,
    COUNT(p.payout_id) AS total_payouts,
    SUM(CASE WHEN p.payout_status = 'Failed' THEN 1 ELSE 0 END) AS failed_payouts,
    ROUND(100.0 * SUM(CASE WHEN p.payout_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(p.payout_id), 2) AS failure_rate_pct
FROM payouts p
JOIN country_master c 
  ON p.receiver_country = c.country
GROUP BY c.region, c.aml_risk
ORDER BY failure_rate_pct DESC;
"""
df_q5 = pd.read_sql(query_5, conn)
df_q5

,region,aml_risk,total_payouts,failed_payouts,failure_rate_pct
0,Africa,High,1969,252,12.80
1,Americas,Medium,3990,213,5.34
2,Africa,Medium,3997,211,5.28
3,APAC,Medium,3984,208,5.22
4,Europe,Low,6093,119,1.95
5,Middle East,Low,6003,107,1.78
6,APAC,Low,9936,158,1.59
7,Americas,Low,4028,63,1.56


### Key Insights

- High-risk African countries have the highest payout failure rate (12.80%).
- Low-risk regions consistently show lower failure rates (around 1.5–2%).
- Higher AML risk is associated with a greater likelihood of payout failures.

### Business Question 6: How Does Compliance Review Impact SLA Performance?

### Objective

Compare SLA performance for transactions that required compliance review versus those that did not, and measure the impact on processing delays.

In [11]:
query_6 = """
SELECT 
    p.payment_type,
    p.compliance_review,
    COUNT(p.payout_id) AS total_transactions,
    SUM(CASE WHEN p.processing_minutes > p.sla_target_minutes THEN 1 ELSE 0 END) AS sla_breaches,
    ROUND(100.0 * SUM(CASE WHEN p.processing_minutes > p.sla_target_minutes THEN 1 ELSE 0 END) / COUNT(p.payout_id), 2) AS breach_rate_pct,
    ROUND(AVG(p.processing_minutes - p.sla_target_minutes), 2) AS avg_delay_minutes
FROM payouts p
GROUP BY p.payment_type, p.compliance_review
ORDER BY breach_rate_pct DESC;
"""
df_q6 = pd.read_sql(query_6, conn)
df_q6

,payment_type,compliance_review,total_transactions,sla_breaches,breach_rate_pct,avg_delay_minutes
0,Education,Yes,751,751,100.00,68.75
1,Family Support,Yes,2566,2566,100.00,66.35
2,Refund,Yes,750,750,100.00,67.78
3,Vendor,Yes,1514,1514,100.00,65.74
4,Salary,Yes,1807,1806,99.94,66.61
5,Refund,No,3266,2027,62.06,8.46
6,Education,No,3212,1984,61.77,8.34
7,Vendor,No,6510,4016,61.69,8.18
8,Family Support,No,11483,7057,61.46,8.03
9,Salary,No,8141,5003,61.45,8.37


### Key Insights

- Transactions requiring compliance review have an SLA breach rate of nearly 100%.
- Transactions without compliance review have breach rates of around 61%.
- Compliance reviews significantly increase processing delays, with average delays exceeding 65 minutes.